# IOAI — 2025 National Selection Three Models (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-national-selection-three-models/data.zip', 'd.zip')
    with zipfile.ZipFile('d.zip') as z: z.extractall('data')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Three Models — 학생 미래 예측: 잃어버린 코드의 비밀 (Georgia National Selection 2025)

조지아 교육연구소의 세 전설적 연구팀이 만든 **3개의 이진 전문가 모델**(scikit-learn LogisticRegression)이
`models.pickle` 에 남아 있다: **Alpha**(Graduate/Dropout), **Beta**(Graduate/Enrolled), **Gamma**(Dropout/Enrolled).
이들을 **똑똑하게 통합**해 학생의 최종 상태를 3분류(Graduate/Dropout/Enrolled)한다. **세 모델의 예측을 반드시
사용**해야 한다(원 데이터만 쓰는 새 분류기로 대체 금지). 채점: held-out **macro-F1**. 제출: `submission.json`={idx: 클래스}.

In [ ]:
import pandas as pd, pickle, json
with open("data/models.pickle", "rb") as f:
    models = pickle.load(f)      # {(classA, classB): LogisticRegression}  (col0->A, col1->B)
train = pd.read_csv("data/train_data.csv")
test  = pd.read_csv("data/test_data.csv")
FEATS = [c for c in train.columns if c != "target"]
print("models:", list(models.keys()), "| train", train.shape, "| test", test.shape)

In [ ]:
# 베이스라인: 전부 "Graduate" (starter code) — macro-F1 ≈ 0.22
def create_predictions_json(test_data):
    predictions = {str(i): "Graduate" for i in range(len(test_data))}
    with open("submission.json", "w") as f:
        json.dump(predictions, f, indent=4)
    print("saved submission.json", len(predictions))
create_predictions_json(test)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)